In [1]:
import pandas as pd
import pickle

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_openai import (
    OpenAIEmbeddings
)

from langchain_community.vectorstores import (
    FAISS
)

from rank_bm25 import BM25Okapi

/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_19246/2983470511.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import (


In [2]:
cnn_df = pd.read_csv(
    "../notebooks/cnn_processed20k.csv"
)

print(
    "Dataset Shape:",
    cnn_df.shape
)

Dataset Shape: (20003, 13)


In [3]:
from config import *

In [4]:
print(
    "Embedding Model:",
    EMBEDDING_MODEL
)

print(
    "Chunk Size:",
    CHUNK_SIZE
)

print(
    "Chunk Overlap:",
    CHUNK_OVERLAP
)

Embedding Model: text-embedding-3-small
Chunk Size: 500
Chunk Overlap: 50


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

all_chunks = []

for article in cnn_df[
    "clean_article"
]:

    chunks = splitter.split_text(
        str(article)
    )

    all_chunks.extend(
        chunks
    )

print(
    "Total Chunks:",
    len(all_chunks)
)

Total Chunks: 160571


In [6]:
# ============================================================
# CHUNK STATISTICS
# ============================================================

total_words = 0

for chunk in all_chunks:
    
    total_words += len(
        chunk.split()
    )

avg_words = (
    total_words /
    len(all_chunks)
)

print(
    "Average Words Per Chunk:",
    round(avg_words, 2)
)

Average Words Per Chunk: 80.99


In [7]:
estimated_tokens = (
    len(all_chunks)
    *
    avg_words
    *
    1.3
)

print(
    f"Estimated Tokens: {estimated_tokens:,.0f}"
)

Estimated Tokens: 16,905,347


In [8]:
# ============================================================
# CREATE OPENAI EMBEDDINGS
# ============================================================

from dotenv import load_dotenv
import os

load_dotenv()

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL
)

print(
    "Embedding Model Loaded Successfully"
)

Embedding Model Loaded Successfully


In [9]:
# ============================================================
# SAVE CHUNKS
# ============================================================

import pickle

with open(
    "all_chunks.pkl",
    "wb"
) as f:

    pickle.dump(
        all_chunks,
        f
    )

print(
    "Chunks Saved Successfully"
)

Chunks Saved Successfully


In [10]:
print(
    "Total Chunks:",
    len(all_chunks)
)

Total Chunks: 160571


In [11]:
vector_store = FAISS.from_texts(
    all_chunks,
    embeddings
)

In [12]:
# ============================================================
# SAVE FAISS INDEX
# ============================================================

vector_store.save_local(
    "faiss_20k"
)

print(
    "FAISS Saved Successfully"
)

FAISS Saved Successfully


In [13]:
import os

print(
    os.listdir(
        "faiss_20k"
    )
)

['index.faiss', 'index.pkl']


In [14]:
# ============================================================
# TOKENIZE CHUNKS
# ============================================================

tokenized_chunks = [

    chunk.split()

    for chunk in all_chunks
]

print(
    "Tokenization Complete"
)

Tokenization Complete


In [15]:
# ============================================================
# BUILD BM25 INDEX
# ============================================================

from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(
    tokenized_chunks
)

print(
    "BM25 Index Created Successfully"
)

BM25 Index Created Successfully


In [16]:
# ============================================================
# SAVE BM25 INDEX
# ============================================================

import pickle

with open(
    "bm25_20k.pkl",
    "wb"
) as f:

    pickle.dump(
        bm25,
        f
    )

print(
    "BM25 Saved Successfully"
)

BM25 Saved Successfully
